# Khối B — train YOLOv8s 6 lớp trên VAIPE (Kaggle T4×2)

Plan §5.2. Bật **GPU T4 x2** + **Internet**. Add dataset VAIPE (thư mục `1_vaipe` có train/valid/test).
Chạy hết (~2h30'). Tải `yolo_vaipe.pt` ở Output về `code/weights/`.

In [ ]:
!pip install -q ultralytics

In [ ]:
import json, os, glob, shutil
from pathlib import Path

# Đổi nếu dataset Kaggle đặt tên khác. Cần cấu trúc <root>/{train,valid,test}/{images,labels}
CANDIDATES = glob.glob("/kaggle/input/*/1_vaipe") + glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
VAIPE = next(Path(c) for c in CANDIDATES if (Path(c) / "train" / "labels").exists())
WORK = Path("/kaggle/working")
LABELS = ["drugname", "quantity", "usage", "diagnose", "date", "other"]
print("VAIPE =", VAIPE, {s: len(list((VAIPE / s / "images").glob("*.png"))) for s in ["train", "valid", "test"]})

In [ ]:
# Chuyển nhãn VAIPE (xyxy pixel) → YOLO (cls cx cy w h chuẩn hóa). Giống code/tools/to_yolo.py
from PIL import Image
Y = WORK / "yolo"
for split in ["train", "valid", "test"]:
    (Y / split / "images").mkdir(parents=True, exist_ok=True)
    (Y / split / "labels").mkdir(parents=True, exist_ok=True)
    for img_path in sorted((VAIPE / split / "images").glob("*.png")):
        dst = Y / split / "images" / img_path.name
        if not dst.exists():
            os.symlink(img_path, dst)
        W, H = Image.open(img_path).size
        lbl = VAIPE / split / "labels" / f"{img_path.stem}.json"
        lines = []
        for r in (json.loads(lbl.read_text(encoding="utf-8")) if lbl.exists() else []):
            if r.get("label") not in LABELS: continue
            x0, y0, x1, y1 = r["box"]
            if x1 <= x0 or y1 <= y0: continue
            lines.append(f"{LABELS.index(r['label'])} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
        (Y / split / "labels" / f"{img_path.stem}.txt").write_text("\n".join(lines))
(Y / "data.yaml").write_text(f"path: {Y}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames: {dict(enumerate(LABELS))}\n")
print((Y / "data.yaml").read_text())

In [ ]:
# Plan §5.2: fliplr=0 (lật làm hỏng chữ), imgsz 1024 giữ chữ nhỏ 15–25 px, seed 42.
# Trễ quá: đổi model="yolov8n.pt" hoặc epochs=30.
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
model.train(data=str(Y / "data.yaml"), imgsz=1024, epochs=50, batch=16, patience=15, seed=42,
            fliplr=0.0, mosaic=0.5, device=[0, 1], workers=4, project=str(WORK / "runs"), name="vaipe", exist_ok=True)

In [ ]:
# Đánh giá: valid (chọn conf) và test (một lần, số liệu E-B2: mAP50 + P/R từng lớp)
best = WORK / "runs/vaipe/weights/best.pt"
m = YOLO(str(best))
for split in ["val", "test"]:
    r = m.val(data=str(Y / "data.yaml"), split=split, imgsz=1024, batch=16, conf=0.001, plots=True,
              project=str(WORK / "runs"), name=f"eval_{split}", exist_ok=True)
    print(split, "mAP50 =", r.box.map50, "mAP50-95 =", r.box.map)
shutil.copy(best, WORK / "yolo_vaipe.pt")
print("→ tải /kaggle/working/yolo_vaipe.pt về code/weights/yolo_vaipe.pt")